In [1]:
import os
import random
import shutil
import tarfile
from pathlib import Path

project_root = Path.cwd()
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("MPLCONFIGDIR", str((project_root / "outputs" / "mpl_config").resolve()))

import h5py
import numpy as np
import pandas as pd
import requests
import torch
import yaml
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

random_seed = 69
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

if not getattr(torch.Tensor.view, "_safe_reshape_fallback_enabled", False):
    _original_tensor_view = torch.Tensor.view

    def safe_tensor_view(self, *shape):
        try:
            return _original_tensor_view(self, *shape)
        except RuntimeError as runtime_error:
            if "view size is not compatible with input tensor's size and stride" in str(runtime_error):
                return self.reshape(*shape)
            raise

    safe_tensor_view._safe_reshape_fallback_enabled = True
    torch.Tensor.view = safe_tensor_view

if torch.cuda.is_available():
    training_device = 0
else:
    training_device = "cpu"

data_root = project_root / "data"
outputs_root = project_root / "outputs"
photos_root = project_root / "photos"
svhn_root = data_root / "svhn"
svhn_raw_root = svhn_root / "raw"
svhn_yolo_root = svhn_root / "yolo_number_boxes"
numberdetection_root = project_root / "numberdetection"

for directory_path in [
    data_root,
    outputs_root,
    photos_root,
    svhn_raw_root,
    svhn_yolo_root,
    numberdetection_root,
]:
    directory_path.mkdir(parents=True, exist_ok=True)

official_dataset_url_by_split = {
    "train": "http://ufldl.stanford.edu/housenumbers/train.tar.gz",
    "test": "http://ufldl.stanford.edu/housenumbers/test.tar.gz",
}

supported_image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
base_model_name = "yolo11n.pt"
image_size = 512
training_batch_size = 8 if training_device == "cpu" else 16
training_worker_count = 0
pretraining_epochs = 10
fine_tuning_epochs = 15
pretraining_patience = 4
fine_tuning_patience = 4
train_validation_fraction = 0.1
test_confidence_threshold = 0.25
photo_inference_confidence = 0.20
photo_inference_size = 1280


/Users/ol.chistov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def list_image_paths(directory_path):
    directory_path = Path(directory_path)
    if not directory_path.exists():
        return []
    return sorted(
        path for path in directory_path.iterdir() if path.is_file() and path.suffix.lower() in supported_image_suffixes
    )


def download_file(file_url, destination_path):
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if destination_path.exists():
        return destination_path

    with requests.get(file_url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))
        with open(destination_path, "wb") as output_file, tqdm(
            total=total_size,
            desc=destination_path.name,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as progress_bar:
            for file_chunk in response.iter_content(chunk_size=1024 * 1024):
                if file_chunk:
                    output_file.write(file_chunk)
                    progress_bar.update(len(file_chunk))

    return destination_path


def extract_archive(archive_path, destination_directory):
    destination_directory.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:*") as archive_file:
        archive_file.extractall(destination_directory)


def read_svhn_attribute(digit_struct_file, bounding_box_dataset, attribute_name):
    attribute_dataset = bounding_box_dataset[attribute_name]
    if attribute_dataset.shape[0] > 1:
        return [
            float(digit_struct_file[attribute_dataset[index][0]][()][0][0])
            for index in range(attribute_dataset.shape[0])
        ]
    return [float(attribute_dataset[()][0][0])]


def load_svhn_annotations(split_directory):
    digit_struct_path = split_directory / "digitStruct.mat"
    annotation_rows = []

    with h5py.File(digit_struct_path, "r") as digit_struct_file:
        image_name_dataset = digit_struct_file["digitStruct"]["name"]
        bounding_box_dataset = digit_struct_file["digitStruct"]["bbox"]

        for row_index in tqdm(range(len(image_name_dataset)), desc=f"{split_directory.name}_annotations"):
            image_name_reference = image_name_dataset[row_index][0]
            image_name = "".join(chr(character[0]) for character in digit_struct_file[image_name_reference][()])
            bounding_box_reference = bounding_box_dataset[row_index][0]
            current_bounding_box_dataset = digit_struct_file[bounding_box_reference]

            left_values = read_svhn_attribute(digit_struct_file, current_bounding_box_dataset, "left")
            top_values = read_svhn_attribute(digit_struct_file, current_bounding_box_dataset, "top")
            width_values = read_svhn_attribute(digit_struct_file, current_bounding_box_dataset, "width")
            height_values = read_svhn_attribute(digit_struct_file, current_bounding_box_dataset, "height")

            image_path = split_directory / image_name
            with Image.open(image_path) as image_file:
                image_width, image_height = image_file.size

            left_edge = max(0.0, min(left_values))
            top_edge = max(0.0, min(top_values))
            right_edge = min(
                float(image_width),
                max(left_value + width_value for left_value, width_value in zip(left_values, width_values)),
            )
            bottom_edge = min(
                float(image_height),
                max(top_value + height_value for top_value, height_value in zip(top_values, height_values)),
            )

            annotation_rows.append(
                {
                    "image_name": image_name,
                    "image_path": image_path,
                    "image_width": image_width,
                    "image_height": image_height,
                    "left_edge": left_edge,
                    "top_edge": top_edge,
                    "right_edge": right_edge,
                    "bottom_edge": bottom_edge,
                }
            )

    return annotation_rows


def link_or_copy_file(source_path, destination_path):
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if destination_path.exists() or destination_path.is_symlink():
        return

    try:
        os.symlink(source_path.resolve(), destination_path)
    except OSError:
        shutil.copy2(source_path, destination_path)


def write_number_label_file(annotation_row, destination_path):
    box_width = annotation_row["right_edge"] - annotation_row["left_edge"]
    box_height = annotation_row["bottom_edge"] - annotation_row["top_edge"]
    x_center = annotation_row["left_edge"] + box_width / 2
    y_center = annotation_row["top_edge"] + box_height / 2

    normalized_x_center = x_center / annotation_row["image_width"]
    normalized_y_center = y_center / annotation_row["image_height"]
    normalized_box_width = box_width / annotation_row["image_width"]
    normalized_box_height = box_height / annotation_row["image_height"]

    with open(destination_path, "w", encoding="utf-8") as output_file:
        output_file.write(
            f"0 {normalized_x_center:.6f} {normalized_y_center:.6f} {normalized_box_width:.6f} {normalized_box_height:.6f}\n"
        )


def prepare_svhn_yolo_dataset():
    dataset_yaml_path = svhn_yolo_root / "dataset.yaml"
    if dataset_yaml_path.exists():
        return dataset_yaml_path

    split_directory_by_name = {}
    for split_name, dataset_url in official_dataset_url_by_split.items():
        archive_path = download_file(dataset_url, svhn_raw_root / f"{split_name}.tar.gz")
        split_directory = svhn_raw_root / split_name
        if not (split_directory / "digitStruct.mat").exists():
            extract_archive(archive_path, svhn_raw_root)
        split_directory_by_name[split_name] = split_directory

    train_annotation_rows = load_svhn_annotations(split_directory_by_name["train"])
    test_annotation_rows = load_svhn_annotations(split_directory_by_name["test"])

    shuffled_train_rows = train_annotation_rows.copy()
    random.Random(random_seed).shuffle(shuffled_train_rows)
    validation_count = int(len(shuffled_train_rows) * train_validation_fraction)
    validation_image_name_set = {row["image_name"] for row in shuffled_train_rows[:validation_count]}

    annotation_rows_by_split = {
        "train": [row for row in train_annotation_rows if row["image_name"] not in validation_image_name_set],
        "valid": [row for row in train_annotation_rows if row["image_name"] in validation_image_name_set],
        "test": test_annotation_rows,
    }

    for split_name, split_rows in annotation_rows_by_split.items():
        image_directory = svhn_yolo_root / split_name / "images"
        label_directory = svhn_yolo_root / split_name / "labels"
        image_directory.mkdir(parents=True, exist_ok=True)
        label_directory.mkdir(parents=True, exist_ok=True)

        for annotation_row in tqdm(split_rows, desc=f"build_{split_name}"):
            destination_image_path = image_directory / annotation_row["image_name"]
            destination_label_path = label_directory / f"{Path(annotation_row['image_name']).stem}.txt"
            link_or_copy_file(annotation_row["image_path"], destination_image_path)
            write_number_label_file(annotation_row, destination_label_path)

    dataset_configuration = {
        "path": str(svhn_yolo_root.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "names": {0: "number"},
    }
    with open(dataset_yaml_path, "w", encoding="utf-8") as output_file:
        yaml.safe_dump(dataset_configuration, output_file, allow_unicode=True, sort_keys=False)

    return dataset_yaml_path


def find_numberdetection_dataset_directory():
    required_split_names = ["train", "valid", "test"]
    if all((numberdetection_root / split_name / "images").exists() for split_name in required_split_names):
        return numberdetection_root

    for candidate_yaml_path in sorted(numberdetection_root.rglob("data.yaml")):
        candidate_directory = candidate_yaml_path.parent
        if all((candidate_directory / split_name / "images").exists() for split_name in required_split_names):
            return candidate_directory

    return None


def prepare_numberdetection_dataset():
    numberdetection_dataset_directory = find_numberdetection_dataset_directory()

    source_yaml_path = numberdetection_dataset_directory / "data.yaml"
    source_configuration = {}
    if source_yaml_path.exists():
        with open(source_yaml_path, "r", encoding="utf-8") as input_file:
            source_configuration = yaml.safe_load(input_file) or {}

    class_names = source_configuration.get(
        "names",
        [str(class_index) for class_index in range(source_configuration.get("nc", 10))],
    )

    normalized_dataset_configuration = {
        "path": str(numberdetection_dataset_directory.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": source_configuration.get("nc", len(class_names)),
        "names": class_names,
    }

    normalized_yaml_path = outputs_root / "numberdetection_local_data.yaml"
    with open(normalized_yaml_path, "w", encoding="utf-8") as output_file:
        yaml.safe_dump(normalized_dataset_configuration, output_file, allow_unicode=True, sort_keys=False)

    return normalized_yaml_path


def read_yaml_configuration(yaml_path):
    with open(yaml_path, "r", encoding="utf-8") as input_file:
        return yaml.safe_load(input_file) or {}


def resolve_dataset_root(dataset_yaml_path):
    dataset_configuration = read_yaml_configuration(dataset_yaml_path)
    dataset_root = dataset_configuration.get("path")
    
    return Path(dataset_root)


def resolve_weight_path(run_directory):
    weights_directory = Path(run_directory) / "weights"
    for weight_file_name in ["best.pt", "last.pt"]:
        candidate_path = weights_directory / weight_file_name
        if candidate_path.exists():
            return candidate_path
    return None


def read_first_yolo_box(label_path, image_width, image_height):
    if not label_path.exists():
        return None

    with open(label_path, "r", encoding="utf-8") as input_file:
        first_line = input_file.readline().strip()

    if not first_line:
        return None

    _, x_center, y_center, box_width, box_height = map(float, first_line.split())
    left_edge = (x_center - box_width / 2) * image_width
    top_edge = (y_center - box_height / 2) * image_height
    right_edge = (x_center + box_width / 2) * image_width
    bottom_edge = (y_center + box_height / 2) * image_height
    return [left_edge, top_edge, right_edge, bottom_edge]


def compute_intersection_over_union(first_box, second_box):
    intersection_left = max(first_box[0], second_box[0])
    intersection_top = max(first_box[1], second_box[1])
    intersection_right = min(first_box[2], second_box[2])
    intersection_bottom = min(first_box[3], second_box[3])

    intersection_width = max(0.0, intersection_right - intersection_left)
    intersection_height = max(0.0, intersection_bottom - intersection_top)
    intersection_area = intersection_width * intersection_height

    first_box_area = max(0.0, first_box[2] - first_box[0]) * max(0.0, first_box[3] - first_box[1])
    second_box_area = max(0.0, second_box[2] - second_box[0]) * max(0.0, second_box[3] - second_box[1])
    union_area = first_box_area + second_box_area - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


def compute_mean_iou(trained_model, image_directory, label_directory, confidence_threshold, inference_device, inference_batch_size):
    iou_values = []

    prediction_stream = trained_model.predict(
        source=str(image_directory),
        imgsz=image_size,
        conf=confidence_threshold,
        batch=inference_batch_size,
        device=inference_device,
        stream=True,
        verbose=False,
    )

    for prediction_result in tqdm(prediction_stream, desc="mean_iou"):
        image_path = Path(prediction_result.path)
        image_height, image_width = prediction_result.orig_shape
        label_path = label_directory / f"{image_path.stem}.txt"
        ground_truth_box = read_first_yolo_box(label_path, image_width, image_height)

        if ground_truth_box is None:
            continue

        if prediction_result.boxes is None or len(prediction_result.boxes) == 0:
            iou_values.append(0.0)
            continue

        predicted_box_array = prediction_result.boxes.xyxy.cpu().numpy()
        best_iou_value = max(
            compute_intersection_over_union(ground_truth_box, predicted_box.tolist())
            for predicted_box in predicted_box_array
        )
        iou_values.append(best_iou_value)

    if not iou_values:
        return 0.0

    return float(np.mean(iou_values))


In [3]:
svhn_dataset_yaml_path = prepare_svhn_yolo_dataset()
numberdetection_dataset_yaml_path = prepare_numberdetection_dataset()
numberdetection_dataset_directory = resolve_dataset_root(numberdetection_dataset_yaml_path)

svhn_overview_table = pd.DataFrame(
    [
        {"split_name": split_name, "image_count": len(list_image_paths(svhn_yolo_root / split_name / "images"))}
        for split_name in ["train", "valid", "test"]
    ]
)
numberdetection_overview_table = pd.DataFrame(
    [
        {
            "split_name": split_name,
            "image_count": len(list_image_paths(numberdetection_dataset_directory / split_name / "images")),
        }
        for split_name in ["train", "valid", "test"]
    ]
)

display(svhn_overview_table)
display(numberdetection_overview_table)
print(f"numberdetection_eppfj локально подключен: {numberdetection_dataset_directory}")


,split_name,image_count
0,train,30062
1,valid,3340
2,test,13068


,split_name,image_count
0,train,1032
1,valid,99
2,test,50


numberdetection_eppfj локально подключен: /Users/ol.chistov/uni_cv/lab2/numberdetection


In [ ]:
runtime_training_device = 0 if torch.cuda.is_available() else "cpu"
runtime_training_batch_size = 8 if runtime_training_device == "cpu" else 16
runtime_training_worker_count = 0

number_pretrain_run_directory = outputs_root / "number_pretrain"
svhn_finetune_run_directory = outputs_root / "svhn_finetune"

number_pretrain_weight_path = resolve_weight_path(number_pretrain_run_directory)
if number_pretrain_weight_path is None:
    number_pretrain_model = YOLO(base_model_name)
    number_pretrain_model.train(
        data=str(numberdetection_dataset_yaml_path),
        epochs=pretraining_epochs,
        imgsz=image_size,
        batch=runtime_training_batch_size,
        device=runtime_training_device,
        project=str(outputs_root),
        name="number_pretrain",
        exist_ok=True,
        seed=random_seed,
        patience=pretraining_patience,
        amp=False,
        workers=runtime_training_worker_count,
        verbose=True,
    )

    number_pretrain_weight_path = resolve_weight_path(number_pretrain_run_directory)


initial_model_source = str(number_pretrain_weight_path)

svhn_finetune_weight_path = resolve_weight_path(svhn_finetune_run_directory)
if svhn_finetune_weight_path is None:
    svhn_finetune_model = YOLO(initial_model_source)
    svhn_finetune_model.train(
        data=str(svhn_dataset_yaml_path),
        epochs=fine_tuning_epochs,
        imgsz=image_size,
        batch=runtime_training_batch_size,
        device=runtime_training_device,
        project=str(outputs_root),
        name="svhn_finetune",
        exist_ok=True,
        seed=random_seed,
        patience=fine_tuning_patience,
        amp=False,
        close_mosaic=5,
        workers=runtime_training_worker_count,
        verbose=True,
    )

    svhn_finetune_weight_path = resolve_weight_path(svhn_finetune_run_directory)


best_model_path = svhn_finetune_weight_path
trained_model = YOLO(str(best_model_path))
print(f"Лучшая модель: {best_model_path}")


Ultralytics 8.4.41 🚀 Python-3.9.6 torch-2.5.1 CPU (Apple M3 Max)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ol.chistov/uni_cv/lab2/outputs/numberdetection_local_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=number_pretrain, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

Matplotlib is building the font cache; this may take a moment.


Overriding model.yaml nc=80 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

In [5]:
runtime_training_device = 0 if torch.cuda.is_available() else "cpu"
runtime_training_batch_size = 8 if runtime_training_device == "cpu" else 16
runtime_training_worker_count = 0

test_metrics = trained_model.val(
    data=str(svhn_dataset_yaml_path),
    split="test",
    imgsz=image_size,
    batch=runtime_training_batch_size,
    device=runtime_training_device,
    workers=runtime_training_worker_count,
    amp=False,
    plots=False,
    verbose=False,
)

mean_iou_value = compute_mean_iou(
    trained_model=trained_model,
    image_directory=svhn_yolo_root / "test" / "images",
    label_directory=svhn_yolo_root / "test" / "labels",
    confidence_threshold=test_confidence_threshold,
    inference_device=runtime_training_device,
    inference_batch_size=runtime_training_batch_size,
)

test_metrics_table = pd.DataFrame(
    [
        {"metric_name": "mean_iou", "metric_value": round(mean_iou_value, 4)},
        {"metric_name": "precision", "metric_value": round(float(test_metrics.box.mp), 4)},
        {"metric_name": "recall", "metric_value": round(float(test_metrics.box.mr), 4)},
        {"metric_name": "map50", "metric_value": round(float(test_metrics.box.map50), 4)},
        {"metric_name": "map50_95", "metric_value": round(float(test_metrics.box.map), 4)},
    ]
)
test_metrics_table.to_csv(outputs_root / "test_metrics.csv", index=False)
display(test_metrics_table)

Ultralytics 8.4.41 🚀 Python-3.9.6 torch-2.5.1 CPU (Apple M3 Max)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 9.3±5.7 MB/s, size: 5.7 KB)
val: Scanning /Users/ol.chistov/uni_cv/lab2/data/svhn/yolo_number_boxes/test/labels... 13068 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13068/13068 4.0Kit/s 3.3s0.0s
val: New cache created: /Users/ol.chistov/uni_cv/lab2/data/svhn/yolo_number_boxes/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1634/1634 11.6it/s 2:210.2ss
                   all      13068      13068      0.896      0.885      0.921      0.533
Speed: 0.1ms preprocess, 8.7ms inference, 0.0ms loss, 0.2ms postprocess per image


mean_iou: 13068it [05:50, 37.33it/s]


,metric_name,metric_value
0,mean_iou,0.7321
1,precision,0.8960
2,recall,0.8853
3,map50,0.9215
4,map50_95,0.5326


In [8]:
runtime_training_device = 0 if torch.cuda.is_available() else "cpu"
photo_image_paths = list_image_paths(photos_root)
photo_results_table = pd.DataFrame(columns=["file_name", "detected_number_count", "max_confidence"])

if photo_image_paths:
    photo_prediction_results = trained_model.predict(
        source=[str(path) for path in photo_image_paths],
        conf=photo_inference_confidence,
        imgsz=photo_inference_size,
        device=runtime_training_device,
        save=True,
        project=str(outputs_root),
        name="photo_predictions",
        exist_ok=True,
        verbose=False,
    )
    photo_rows = []
    for prediction_result in photo_prediction_results:
        detected_number_count = 0 if prediction_result.boxes is None else len(prediction_result.boxes)
        max_confidence = 0.0
        if detected_number_count:
            max_confidence = float(prediction_result.boxes.conf.max().item())
        photo_rows.append(
            {
                "file_name": Path(prediction_result.path).name,
                "detected_number_count": detected_number_count,
                "max_confidence": round(max_confidence, 4),
            }
        )
    photo_results_table = pd.DataFrame(photo_rows).sort_values("file_name").reset_index(drop=True)
else:
    print(f"В папке {photos_root} не найдено фотографий для инференса")

photo_results_table.to_csv(outputs_root / "photo_predictions.csv", index=False)

display(photo_results_table)


Results saved to /Users/ol.chistov/uni_cv/lab2/outputs/photo_predictions


,file_name,detected_number_count,max_confidence
0,image0.jpg,3,0.4801
1,image1.jpg,3,0.7481
2,image2.jpg,3,0.5370
3,image3.jpg,3,0.7351
4,image4.jpg,3,0.7844
5,image5.jpg,3,0.6707
6,image6.jpg,4,0.7091


In [9]:

conclusion_lines = [
    f"device: {training_device}",
    f"best_model_path: {best_model_path}",
    f"numberdetection_dataset_yaml_path: {numberdetection_dataset_yaml_path}",
    f"mean_iou: {mean_iou_value:.4f}",
    f"precision: {float(test_metrics.box.mp):.4f}",
    f"recall: {float(test_metrics.box.mr):.4f}",
    f"map50: {float(test_metrics.box.map50):.4f}",
    f"map50_95: {float(test_metrics.box.map):.4f}",
    "numberdetection_pretraining: used",
]

if photo_image_paths:
    conclusion_lines.append(f"photo_predictions_path: {outputs_root / 'photo_predictions'}")
else:
    conclusion_lines.append(f"photos_directory: {photos_root}")

print("\n".join(conclusion_lines))


device: cpu
best_model_path: /Users/ol.chistov/uni_cv/lab2/outputs/svhn_finetune/weights/best.pt
numberdetection_dataset_yaml_path: /Users/ol.chistov/uni_cv/lab2/outputs/numberdetection_local_data.yaml
mean_iou: 0.7321
precision: 0.8960
recall: 0.8853
map50: 0.9215
map50_95: 0.5326
numberdetection_pretraining: used
photo_predictions_path: /Users/ol.chistov/uni_cv/lab2/outputs/photo_predictions
